# 01 - Data Audit and Cleaning

This notebook implements Stage 1 of the credit-risk pipeline using the official Home Credit dataset.

It performs:
- raw table audit for the six core tables
- duplicate and missingness checks
- key anomaly handling
- derived cleaning flags for repayment behavior
- cleaned parquet outputs for downstream modeling
- audit summary artifacts saved under `artifacts/raw_checks/`


In [1]:
from __future__ import annotations

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.options.display.max_columns = 200
pd.options.display.max_rows = 200

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_DIR = ROOT / 'data' / 'home_credit_full'
RAW_CHECKS_DIR = ROOT / 'artifacts' / 'raw_checks'
PROCESSED_DIR = ROOT / 'artifacts' / 'processed'

RAW_CHECKS_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

CORE_FILES = [
    'application_train.csv',
    'installments_payments.csv',
    'POS_CASH_balance.csv',
    'previous_application.csv',
    'bureau.csv',
    'bureau_balance.csv',
]

for name in CORE_FILES:
    path = DATA_DIR / name
    if not path.exists():
        raise FileNotFoundError(f'Missing required file: {path}')

print(f'Using data directory: {DATA_DIR}')
print('Files verified successfully.')


Using data directory: Y:\College\ML\data\home_credit_full
Files verified successfully.


In [2]:
APPLICATION_COLS = [
    'SK_ID_CURR', 'TARGET', 'CODE_GENDER', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE',
    'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'AMT_INCOME_TOTAL',
    'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'DAYS_BIRTH', 'DAYS_EMPLOYED',
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3'
]

INSTALLMENT_COLS = [
    'SK_ID_PREV', 'SK_ID_CURR', 'NUM_INSTALMENT_VERSION', 'NUM_INSTALMENT_NUMBER',
    'DAYS_INSTALMENT', 'DAYS_ENTRY_PAYMENT', 'AMT_INSTALMENT', 'AMT_PAYMENT'
]

POS_COLS = [
    'SK_ID_PREV', 'SK_ID_CURR', 'MONTHS_BALANCE', 'CNT_INSTALMENT', 'CNT_INSTALMENT_FUTURE',
    'NAME_CONTRACT_STATUS', 'SK_DPD', 'SK_DPD_DEF'
]

PREVIOUS_COLS = [
    'SK_ID_PREV', 'SK_ID_CURR', 'NAME_CONTRACT_TYPE', 'NAME_CONTRACT_STATUS',
    'AMT_ANNUITY', 'AMT_APPLICATION', 'AMT_CREDIT', 'AMT_DOWN_PAYMENT',
    'DAYS_DECISION', 'NAME_YIELD_GROUP'
]

BUREAU_COLS = [
    'SK_ID_CURR', 'SK_ID_BUREAU', 'CREDIT_ACTIVE', 'CREDIT_CURRENCY', 'DAYS_CREDIT',
    'DAYS_CREDIT_ENDDATE', 'DAYS_ENDDATE_FACT', 'CREDIT_DAY_OVERDUE', 'AMT_CREDIT_SUM',
    'AMT_CREDIT_SUM_DEBT', 'AMT_CREDIT_MAX_OVERDUE'
]

BUREAU_BALANCE_COLS = ['SK_ID_BUREAU', 'MONTHS_BALANCE', 'STATUS']

def write_markdown(path: Path, text: str) -> None:
    path.write_text(text, encoding='utf-8')

def missing_summary(df: pd.DataFrame, table_name: str) -> pd.DataFrame:
    summary = pd.DataFrame({
        'table_name': table_name,
        'column_name': df.columns,
        'missing_count': df.isna().sum().values,
    })
    summary['missing_pct'] = summary['missing_count'] / len(df)
    return summary.sort_values(['missing_pct', 'column_name'], ascending=[False, True]).reset_index(drop=True)

def summarize_numeric(series: pd.Series) -> dict:
    clean = series.dropna()
    if clean.empty:
        return {'min': np.nan, 'max': np.nan, 'mean': np.nan}
    return {'min': float(clean.min()), 'max': float(clean.max()), 'mean': float(clean.mean())}

def previous_status_group(series: pd.Series) -> pd.Series:
    approved = {'Approved'}
    rejected = {'Refused'}
    active = {'Active'}
    unused = {'Unused offer'}
    canceled = {'Canceled'}
    mapping = {}
    for value in series.dropna().unique():
        if value in approved:
            mapping[value] = 'approved'
        elif value in rejected:
            mapping[value] = 'rejected'
        elif value in active:
            mapping[value] = 'active'
        elif value in unused:
            mapping[value] = 'unused_offer'
        elif value in canceled:
            mapping[value] = 'canceled'
        else:
            mapping[value] = 'other'
    return series.map(mapping).fillna('missing')

def bureau_status_group(series: pd.Series) -> pd.Series:
    mapping = {
        '0': 'current',
        '1': 'dpd_1_30',
        '2': 'dpd_31_60',
        '3': 'dpd_61_90',
        '4': 'dpd_91_120',
        '5': 'dpd_120_plus',
        'C': 'closed',
        'X': 'unknown_or_no_loan',
    }
    return series.map(mapping).fillna('other')

audit_rows = []
missing_frames = []
duplicate_rows = []
anomaly_notes = []


In [3]:
app = pd.read_csv(DATA_DIR / 'application_train.csv', usecols=APPLICATION_COLS)

duplicate_sk_curr = int(app.duplicated(subset=['SK_ID_CURR']).sum())
employment_placeholder_count = int((app['DAYS_EMPLOYED'] == 365243).sum())

app['DAYS_EMPLOYED_ANOMALY_FLAG'] = (app['DAYS_EMPLOYED'] == 365243).astype('int8')
app.loc[app['DAYS_EMPLOYED'] == 365243, 'DAYS_EMPLOYED'] = np.nan
app['AGE_YEARS'] = (-app['DAYS_BIRTH'] / 365.25).round(2)
app['EMPLOYMENT_YEARS'] = (-app['DAYS_EMPLOYED'] / 365.25).round(2)
app['CREDIT_TO_INCOME_RATIO'] = app['AMT_CREDIT'] / app['AMT_INCOME_TOTAL'].replace({0: np.nan})
app['ANNUITY_TO_INCOME_RATIO'] = app['AMT_ANNUITY'] / app['AMT_INCOME_TOTAL'].replace({0: np.nan})
app['GOODS_TO_CREDIT_RATIO'] = app['AMT_GOODS_PRICE'] / app['AMT_CREDIT'].replace({0: np.nan})

audit_rows.append({
    'table_name': 'application_train',
    'row_count': len(app),
    'column_count': app.shape[1],
    'duplicate_key_count': duplicate_sk_curr,
    'primary_key': 'SK_ID_CURR',
    'key_missing_count': int(app['SK_ID_CURR'].isna().sum()),
})
duplicate_rows.append({
    'table_name': 'application_train',
    'duplicate_check': 'SK_ID_CURR',
    'duplicate_count': duplicate_sk_curr,
})
missing_frames.append(missing_summary(app, 'application_train'))
anomaly_notes.append(
    f"application_train: replaced {employment_placeholder_count} rows where DAYS_EMPLOYED == 365243 with NaN and retained a flag column."
)

app.to_parquet(PROCESSED_DIR / 'cleaned_application_train.parquet', index=False)
print(app.head())
print(f'Saved: {PROCESSED_DIR / "cleaned_application_train.parquet"}')


   SK_ID_CURR  TARGET CODE_GENDER  AMT_INCOME_TOTAL  AMT_CREDIT  AMT_ANNUITY  \
0      100002       1           M          202500.0    406597.5      24700.5   
1      100003       0           F          270000.0   1293502.5      35698.5   
2      100004       0           M           67500.0    135000.0       6750.0   
3      100006       0           F          135000.0    312682.5      29686.5   
4      100007       0           M          121500.0    513000.0      21865.5   

   AMT_GOODS_PRICE NAME_INCOME_TYPE            NAME_EDUCATION_TYPE  \
0         351000.0          Working  Secondary / secondary special   
1        1129500.0    State servant               Higher education   
2         135000.0          Working  Secondary / secondary special   
3         297000.0          Working  Secondary / secondary special   
4         513000.0          Working  Secondary / secondary special   

     NAME_FAMILY_STATUS  NAME_HOUSING_TYPE  DAYS_BIRTH  DAYS_EMPLOYED  \
0  Single / not married  

In [4]:
inst = pd.read_csv(DATA_DIR / 'installments_payments.csv', usecols=INSTALLMENT_COLS)

installment_dup_keys = ['SK_ID_PREV', 'SK_ID_CURR', 'NUM_INSTALMENT_VERSION', 'NUM_INSTALMENT_NUMBER']
duplicate_installments = int(inst.duplicated(subset=installment_dup_keys).sum())

inst['payment_delay_days'] = inst['DAYS_ENTRY_PAYMENT'] - inst['DAYS_INSTALMENT']
inst['missing_payment_flag'] = (inst['DAYS_ENTRY_PAYMENT'].isna() | inst['AMT_PAYMENT'].isna() | (inst['AMT_PAYMENT'] <= 0)).astype('int8')
inst['partial_payment_flag'] = (
    (inst['AMT_PAYMENT'].fillna(0) > 0) & (inst['AMT_PAYMENT'].fillna(0) + 1e-9 < inst['AMT_INSTALMENT'].fillna(0))
).astype('int8')
inst['late_payment_flag'] = (inst['payment_delay_days'].fillna(0) > 0).astype('int8')

group_cols = installment_dup_keys
inst_events = inst.groupby(group_cols, dropna=False, as_index=False).agg(
    DAYS_INSTALMENT=('DAYS_INSTALMENT', 'first'),
    first_payment_day=('DAYS_ENTRY_PAYMENT', 'min'),
    last_payment_day=('DAYS_ENTRY_PAYMENT', 'max'),
    AMT_INSTALMENT=('AMT_INSTALMENT', 'max'),
    total_amount_paid=('AMT_PAYMENT', 'sum'),
    row_count=('SK_ID_PREV', 'size'),
    missing_payment_flag=('missing_payment_flag', 'max'),
    partial_payment_flag=('partial_payment_flag', 'max'),
    late_payment_flag=('late_payment_flag', 'max'),
    max_payment_delay_days=('payment_delay_days', 'max'),
)
inst_events['aggregated_payment_delay_days'] = inst_events['first_payment_day'] - inst_events['DAYS_INSTALMENT']
inst_events['underpaid_after_aggregation_flag'] = (
    inst_events['total_amount_paid'].fillna(0) + 1e-9 < inst_events['AMT_INSTALMENT'].fillna(0)
).astype('int8')

audit_rows.append({
    'table_name': 'installments_payments',
    'row_count': len(inst),
    'column_count': inst.shape[1],
    'duplicate_key_count': duplicate_installments,
    'primary_key': '|'.join(installment_dup_keys),
    'key_missing_count': int(inst[installment_dup_keys].isna().any(axis=1).sum()),
})
duplicate_rows.append({
    'table_name': 'installments_payments',
    'duplicate_check': '|'.join(installment_dup_keys),
    'duplicate_count': duplicate_installments,
})
missing_frames.append(missing_summary(inst, 'installments_payments'))
anomaly_notes.append(
    'installments_payments: derived payment_delay_days, missing_payment_flag, partial_payment_flag, and aggregated duplicate installment events.'
)

inst.to_parquet(PROCESSED_DIR / 'cleaned_installments_payments.parquet', index=False)
inst_events.to_parquet(PROCESSED_DIR / 'cleaned_installment_events.parquet', index=False)
print(inst_events.head())
print(f'Saved: {PROCESSED_DIR / "cleaned_installments_payments.parquet"}')
print(f'Saved: {PROCESSED_DIR / "cleaned_installment_events.parquet"}')


   SK_ID_PREV  SK_ID_CURR  NUM_INSTALMENT_VERSION  NUM_INSTALMENT_NUMBER  \
0     1000001      158271                     1.0                      1   
1     1000001      158271                     2.0                      2   
2     1000002      101962                     1.0                      1   
3     1000002      101962                     1.0                      2   
4     1000002      101962                     1.0                      3   

   DAYS_INSTALMENT  first_payment_day  last_payment_day  AMT_INSTALMENT  \
0           -268.0             -294.0            -294.0        6404.310   
1           -238.0             -244.0            -244.0       62039.115   
2          -1600.0            -1611.0           -1611.0        6264.000   
3          -1570.0            -1575.0           -1575.0        6264.000   
4          -1540.0            -1559.0           -1559.0        6264.000   

   total_amount_paid  row_count  missing_payment_flag  partial_payment_flag  \
0           6

In [5]:
pos = pd.read_csv(DATA_DIR / 'POS_CASH_balance.csv', usecols=POS_COLS)

pos_dup_keys = ['SK_ID_PREV', 'MONTHS_BALANCE']
duplicate_pos = int(pos.duplicated(subset=pos_dup_keys).sum())
pos['HAS_DPD_FLAG'] = (pos['SK_DPD'].fillna(0) > 0).astype('int8')
pos['HAS_DPD_DEF_FLAG'] = (pos['SK_DPD_DEF'].fillna(0) > 0).astype('int8')

sequence_counts = pos.groupby('SK_ID_PREV')['MONTHS_BALANCE'].agg(['count', 'nunique'])
sequence_issue_count = int((sequence_counts['count'] != sequence_counts['nunique']).sum())

audit_rows.append({
    'table_name': 'POS_CASH_balance',
    'row_count': len(pos),
    'column_count': pos.shape[1],
    'duplicate_key_count': duplicate_pos,
    'primary_key': '|'.join(pos_dup_keys),
    'key_missing_count': int(pos[pos_dup_keys].isna().any(axis=1).sum()),
})
duplicate_rows.append({
    'table_name': 'POS_CASH_balance',
    'duplicate_check': '|'.join(pos_dup_keys),
    'duplicate_count': duplicate_pos,
})
missing_frames.append(missing_summary(pos, 'POS_CASH_balance'))
anomaly_notes.append(
    f'POS_CASH_balance: found {sequence_issue_count} SK_ID_PREV sequences where MONTHS_BALANCE count differs from unique count.'
)

pos.to_parquet(PROCESSED_DIR / 'cleaned_pos_cash_balance.parquet', index=False)
print(pos.head())
print(f'Saved: {PROCESSED_DIR / "cleaned_pos_cash_balance.parquet"}')


   SK_ID_PREV  SK_ID_CURR  MONTHS_BALANCE  CNT_INSTALMENT  \
0     1803195      182943             -31            48.0   
1     1715348      367990             -33            36.0   
2     1784872      397406             -32            12.0   
3     1903291      269225             -35            48.0   
4     2341044      334279             -35            36.0   

   CNT_INSTALMENT_FUTURE NAME_CONTRACT_STATUS  SK_DPD  SK_DPD_DEF  \
0                   45.0               Active       0           0   
1                   35.0               Active       0           0   
2                    9.0               Active       0           0   
3                   42.0               Active       0           0   
4                   35.0               Active       0           0   

   HAS_DPD_FLAG  HAS_DPD_DEF_FLAG  
0             0                 0  
1             0                 0  
2             0                 0  
3             0                 0  
4             0                 0  
Sa

In [6]:
prev = pd.read_csv(DATA_DIR / 'previous_application.csv', usecols=PREVIOUS_COLS)

prev_dup_keys = ['SK_ID_PREV', 'SK_ID_CURR']
duplicate_prev = int(prev.duplicated(subset=prev_dup_keys).sum())
prev['status_group'] = previous_status_group(prev['NAME_CONTRACT_STATUS'])

audit_rows.append({
    'table_name': 'previous_application',
    'row_count': len(prev),
    'column_count': prev.shape[1],
    'duplicate_key_count': duplicate_prev,
    'primary_key': '|'.join(prev_dup_keys),
    'key_missing_count': int(prev[prev_dup_keys].isna().any(axis=1).sum()),
})
duplicate_rows.append({
    'table_name': 'previous_application',
    'duplicate_check': '|'.join(prev_dup_keys),
    'duplicate_count': duplicate_prev,
})
missing_frames.append(missing_summary(prev, 'previous_application'))
anomaly_notes.append('previous_application: added status_group to collapse active/completed/rejected style states.')

prev.to_parquet(PROCESSED_DIR / 'cleaned_previous_application.parquet', index=False)
print(prev.head())
print(f'Saved: {PROCESSED_DIR / "cleaned_previous_application.parquet"}')


   SK_ID_PREV  SK_ID_CURR NAME_CONTRACT_TYPE  AMT_ANNUITY  AMT_APPLICATION  \
0     2030495      271877     Consumer loans     1730.430          17145.0   
1     2802425      108129         Cash loans    25188.615         607500.0   
2     2523466      122040         Cash loans    15060.735         112500.0   
3     2819243      176158         Cash loans    47041.335         450000.0   
4     1784265      202054         Cash loans    31924.395         337500.0   

   AMT_CREDIT  AMT_DOWN_PAYMENT NAME_CONTRACT_STATUS  DAYS_DECISION  \
0     17145.0               0.0             Approved            -73   
1    679671.0               NaN             Approved           -164   
2    136444.5               NaN             Approved           -301   
3    470790.0               NaN             Approved           -512   
4    404055.0               NaN              Refused           -781   

  NAME_YIELD_GROUP status_group  
0           middle     approved  
1       low_action     approved  
2 

In [7]:
bureau = pd.read_csv(DATA_DIR / 'bureau.csv', usecols=BUREAU_COLS)
bureau_balance = pd.read_csv(DATA_DIR / 'bureau_balance.csv', usecols=BUREAU_BALANCE_COLS)

duplicate_bureau = int(bureau.duplicated(subset=['SK_ID_BUREAU']).sum())
duplicate_bureau_balance = int(bureau_balance.duplicated(subset=['SK_ID_BUREAU', 'MONTHS_BALANCE']).sum())

bureau_balance['status_group'] = bureau_status_group(bureau_balance['STATUS'])

bureau_keys = set(bureau['SK_ID_BUREAU'].dropna().astype('int64').tolist())
bureau_balance_keys = set(bureau_balance['SK_ID_BUREAU'].dropna().astype('int64').tolist())
linked_key_count = len(bureau_keys & bureau_balance_keys)
unmatched_balance_key_count = len(bureau_balance_keys - bureau_keys)

audit_rows.append({
    'table_name': 'bureau',
    'row_count': len(bureau),
    'column_count': bureau.shape[1],
    'duplicate_key_count': duplicate_bureau,
    'primary_key': 'SK_ID_BUREAU',
    'key_missing_count': int(bureau['SK_ID_BUREAU'].isna().sum()),
})
audit_rows.append({
    'table_name': 'bureau_balance',
    'row_count': len(bureau_balance),
    'column_count': bureau_balance.shape[1],
    'duplicate_key_count': duplicate_bureau_balance,
    'primary_key': 'SK_ID_BUREAU|MONTHS_BALANCE',
    'key_missing_count': int(bureau_balance[['SK_ID_BUREAU', 'MONTHS_BALANCE']].isna().any(axis=1).sum()),
})
duplicate_rows.extend([
    {
        'table_name': 'bureau',
        'duplicate_check': 'SK_ID_BUREAU',
        'duplicate_count': duplicate_bureau,
    },
    {
        'table_name': 'bureau_balance',
        'duplicate_check': 'SK_ID_BUREAU|MONTHS_BALANCE',
        'duplicate_count': duplicate_bureau_balance,
    },
])
missing_frames.append(missing_summary(bureau, 'bureau'))
missing_frames.append(missing_summary(bureau_balance, 'bureau_balance'))
anomaly_notes.append(
    f'bureau/bureau_balance: linked {linked_key_count} bureau keys; bureau_balance has {unmatched_balance_key_count} unmatched SK_ID_BUREAU keys.'
)

bureau.to_parquet(PROCESSED_DIR / 'cleaned_bureau.parquet', index=False)
bureau_balance.to_parquet(PROCESSED_DIR / 'cleaned_bureau_balance.parquet', index=False)
print(bureau.head())
print(bureau_balance.head())
print(f'Saved: {PROCESSED_DIR / "cleaned_bureau.parquet"}')
print(f'Saved: {PROCESSED_DIR / "cleaned_bureau_balance.parquet"}')


   SK_ID_CURR  SK_ID_BUREAU CREDIT_ACTIVE CREDIT_CURRENCY  DAYS_CREDIT  \
0      215354       5714462        Closed      currency 1         -497   
1      215354       5714463        Active      currency 1         -208   
2      215354       5714464        Active      currency 1         -203   
3      215354       5714465        Active      currency 1         -203   
4      215354       5714466        Active      currency 1         -629   

   CREDIT_DAY_OVERDUE  DAYS_CREDIT_ENDDATE  DAYS_ENDDATE_FACT  \
0                   0               -153.0             -153.0   
1                   0               1075.0                NaN   
2                   0                528.0                NaN   
3                   0                  NaN                NaN   
4                   0               1197.0                NaN   

   AMT_CREDIT_MAX_OVERDUE  AMT_CREDIT_SUM  AMT_CREDIT_SUM_DEBT  
0                     NaN         91323.0                  0.0  
1                     NaN        2

In [8]:
audit_summary = pd.DataFrame(audit_rows).sort_values('table_name').reset_index(drop=True)
missing_summary_df = pd.concat(missing_frames, ignore_index=True)
duplicate_summary_df = pd.DataFrame(duplicate_rows).sort_values(['table_name', 'duplicate_check']).reset_index(drop=True)

audit_summary.to_csv(RAW_CHECKS_DIR / 'raw_audit_summary.csv', index=False)
missing_summary_df.to_csv(RAW_CHECKS_DIR / 'raw_missingness_summary.csv', index=False)
duplicate_summary_df.to_csv(RAW_CHECKS_DIR / 'raw_duplicate_summary.csv', index=False)
write_markdown(RAW_CHECKS_DIR / 'raw_anomaly_notes.md', '# Raw Anomaly Notes\n\n' + '\n'.join(f'- {note}' for note in anomaly_notes) + '\n')

manifest = {
    'cleaned_outputs': sorted(p.name for p in PROCESSED_DIR.glob('cleaned_*.parquet')),
    'audit_outputs': sorted(p.name for p in RAW_CHECKS_DIR.iterdir()),
}
(RAW_CHECKS_DIR / 'stage1_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')

print('Stage 1 artifacts saved successfully.')
display(audit_summary)
display(duplicate_summary_df)
display(missing_summary_df.groupby('table_name').head(5))


Stage 1 artifacts saved successfully.


,table_name,row_count,column_count,duplicate_key_count,primary_key,key_missing_count
0,POS_CASH_balance,10001358,10,0,SK_ID_PREV|MONTHS_BALANCE,0
1,application_train,307511,23,0,SK_ID_CURR,0
2,bureau,1716428,11,0,SK_ID_BUREAU,0
3,bureau_balance,27299925,4,0,SK_ID_BUREAU|MONTHS_BALANCE,0
4,installments_payments,13605401,12,653483,SK_ID_PREV|SK_ID_CURR|NUM_INSTALMENT_VERSION|N...,0
5,previous_application,1670214,11,0,SK_ID_PREV|SK_ID_CURR,0


,table_name,duplicate_check,duplicate_count
0,POS_CASH_balance,SK_ID_PREV|MONTHS_BALANCE,0
1,application_train,SK_ID_CURR,0
2,bureau,SK_ID_BUREAU,0
3,bureau_balance,SK_ID_BUREAU|MONTHS_BALANCE,0
4,installments_payments,SK_ID_PREV|SK_ID_CURR|NUM_INSTALMENT_VERSION|N...,653483
5,previous_application,SK_ID_PREV|SK_ID_CURR,0


,table_name,column_name,missing_count,missing_pct
0,application_train,EXT_SOURCE_1,173378,5.638107e-01
1,application_train,OCCUPATION_TYPE,96391,3.134555e-01
2,application_train,EXT_SOURCE_3,60965,1.982531e-01
3,application_train,DAYS_EMPLOYED,55374,1.800716e-01
4,application_train,EMPLOYMENT_YEARS,55374,1.800716e-01
23,installments_payments,AMT_PAYMENT,2905,2.135181e-04
24,installments_payments,DAYS_ENTRY_PAYMENT,2905,2.135181e-04
25,installments_payments,payment_delay_days,2905,2.135181e-04
26,installments_payments,AMT_INSTALMENT,0,0.000000e+00
27,installments_payments,DAYS_INSTALMENT,0,0.000000e+00


## Stage 1 Outputs

Saved artifacts:
- `artifacts/raw_checks/raw_audit_summary.csv`
- `artifacts/raw_checks/raw_missingness_summary.csv`
- `artifacts/raw_checks/raw_duplicate_summary.csv`
- `artifacts/raw_checks/raw_anomaly_notes.md`
- `artifacts/raw_checks/stage1_manifest.json`
- `artifacts/processed/cleaned_*.parquet`

These cleaned tables will be the inputs for Stage 2 target construction.
